In [51]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                            roc_auc_score, f1_score, average_precision_score)
import xgboost as xgb
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')



# I. Preprocess Train

In [52]:
df = pd.read_csv('../data/kaggle_b2_fraud_train_v4.csv')  
print(f"✅ {df.shape[0]} lignes × {df.shape[1]} colonnes")


customer_ids = df["customer_id"]


✅ 160000 lignes × 34 colonnes


## 1. Traitement des dates

In [53]:
if 'signup_date' in df.columns:
    df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
    df['signup_year'] = df['signup_date'].dt.year
    df['signup_month'] = df['signup_date'].dt.month
    df['signup_day_of_week'] = df['signup_date'].dt.dayofweek
    df['days_since_signup'] = (datetime.now() - df['signup_date']).dt.days
    df = df.drop('signup_date', axis=1)
    print("    ✓ Features temporelles créées")

    ✓ Features temporelles créées


## 2. Identification des colonnes

In [54]:
print("\n2️⃣  Identification des colonnes...")
exclude_cols = [
    'customer_id', 'account_id', 'customer_note', 'referrer_code',
    'secondary_email', 'postal_code', 'city', 'last_ticket_subject',
    'post_event_status_code', 'chargeback_resolution_time_days',
    'manual_review_result'
]

target = 'target_is_fraud'
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in exclude_cols and c != target]
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in exclude_cols]

print(f"    ✓ {len(numeric_cols)} colonnes numériques")
print(f"    ✓ {len(categorical_cols)} colonnes catégorielles")



2️⃣  Identification des colonnes...
    ✓ 27 colonnes numériques
    ✓ 2 colonnes catégorielles


## 3. Valeurs manquantes

In [55]:
print("\n3️⃣  Traitement des valeurs manquantes...")
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna('Unknown', inplace=True)
print("    ✓ Valeurs manquantes traitées")


3️⃣  Traitement des valeurs manquantes...
    ✓ Valeurs manquantes traitées


## 4. Feature engeneering

In [56]:
print("\n4️⃣  Feature Engineering...")
features_created = 0

if 'num_transactions_30d' in df.columns and 'tenure_months' in df.columns:
    df['tx_per_month'] = df['num_transactions_30d'] / (df['tenure_months'] + 1)
    features_created += 1

if 'avg_amount_30d_eur' in df.columns and 'annual_income_eur' in df.columns:
    df['tx_to_income_ratio'] = df['avg_amount_30d_eur'] / (df['annual_income_eur'] / 12 + 1)
    features_created += 1

if 'chargebacks_12m' in df.columns and 'num_transactions_30d' in df.columns:
    df['chargeback_rate'] = df['chargebacks_12m'] / (df['num_transactions_30d'] * 4 + 1)
    features_created += 1

if 'failed_payments_6m' in df.columns and 'num_transactions_30d' in df.columns:
    df['failed_payment_rate'] = df['failed_payments_6m'] / (df['num_transactions_30d'] * 2 + 1)
    features_created += 1

risk_cols = [c for c in ['chargebacks_12m', 'failed_payments_6m', 'support_tickets_90d',
             'is_vpn', 'is_new_device'] if c in df.columns]
if len(risk_cols) > 0:
    risk_df = df[risk_cols].copy()
    for col in risk_cols:
        if df[col].std() > 0:
            risk_df[col] = (df[col] - df[col].mean()) / df[col].std()
    df['composite_risk_score'] = risk_df.sum(axis=1)
    features_created += 1

print(f"    ✓ {features_created} nouvelles features créées")



4️⃣  Feature Engineering...
    ✓ 5 nouvelles features créées


## 5. Encodage catégorielles

In [57]:
print("\n5️⃣  Encodage des variables catégorielles...")
label_encoders = {}
for col in categorical_cols:
    if col in df.columns:
        if df[col].nunique() > 50:
            top_values = df[col].value_counts().head(50).index
            df[col] = df[col].apply(lambda x: x if x in top_values else 'Other')
        
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

print(f"    ✓ {len(categorical_cols)} colonnes encodées")



5️⃣  Encodage des variables catégorielles...
    ✓ 2 colonnes encodées


## 6. Traitement des outliers

In [58]:
print("\n6️⃣  Traitement des outliers...")
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in exclude_cols and c != target]

for col in numeric_cols:
    if col in df.columns:
        q1 = df[col].quantile(0.01)
        q99 = df[col].quantile(0.99)
        df[col] = df[col].clip(lower=q1, upper=q99)
print(f"    ✓ Outliers traités")




6️⃣  Traitement des outliers...
    ✓ Outliers traités


## 7. Décision finale

In [59]:
features = [c for c in df.columns if c not in exclude_cols ]  
df_features = df[features]

df_processed = df_features.copy()
if 'customer_id' in df.columns:
    df_processed['customer_id'] = df['customer_id']

## 8. Export

In [60]:

print("\n💾 Export du dataset preprocessé...")
df_processed.to_csv('../data/data_preprocess_rf_train_v3.csv', index=False)
print("✅ data_preprocess_rf_test.csv créé!")


💾 Export du dataset preprocessé...
✅ data_preprocess_rf_test.csv créé!


# II. Preprocess Test

In [61]:
df_test = pd.read_csv('../data/kaggle_b2_fraud_test_v4.csv')  
print(f"✅ {df.shape[0]} lignes × {df.shape[1]} colonnes")


customer_ids = df_test["customer_id"]



✅ 160000 lignes × 42 colonnes


## 1. Traitement des dates

In [62]:

if 'signup_date' in df_test.columns:
    df_test['signup_date'] = pd.to_datetime(df_test['signup_date'], errors='coerce')
    df_test['signup_year'] = df_test['signup_date'].dt.year
    df_test['signup_month'] = df_test['signup_date'].dt.month
    df_test['signup_day_of_week'] = df_test['signup_date'].dt.dayofweek
    df_test['days_since_signup'] = (datetime.now() - df_test['signup_date']).dt.days
    df_test = df_test.drop('signup_date', axis=1)
    print("    ✓ Features temporelles créées")



    ✓ Features temporelles créées


## 2. Identification des colonnes

In [63]:
print("\n2️⃣  Identification des colonnes...")
exclude_cols = [
    'customer_id', 'account_id', 'customer_note', 'referrer_code',
    'secondary_email', 'postal_code', 'city', 'last_ticket_subject',
    'post_event_status_code', 'chargeback_resolution_time_days',
    'manual_review_result'
]

numeric_cols = df_test.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in exclude_cols]
categorical_cols = df_test.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in exclude_cols]

print(f"    ✓ {len(numeric_cols)} colonnes numériques")
print(f"    ✓ {len(categorical_cols)} colonnes catégorielles")


2️⃣  Identification des colonnes...
    ✓ 27 colonnes numériques
    ✓ 2 colonnes catégorielles


## 3. Valeurs manquantes

In [64]:
print("\n3️⃣  Traitement des valeurs manquantes...")
for col in numeric_cols:
    if df_test[col].isnull().sum() > 0:
        df_test[col].fillna(df_test[col].median(), inplace=True)

for col in categorical_cols:
    if df_test[col].isnull().sum() > 0:
        df_test[col].fillna('Unknown', inplace=True)
print("    ✓ Valeurs manquantes traitées")



3️⃣  Traitement des valeurs manquantes...
    ✓ Valeurs manquantes traitées


## 4. Feature engeneering

In [65]:
print("\n4️⃣  Feature Engineering...")
features_created = 0

if 'num_transactions_30d' in df_test.columns and 'tenure_months' in df_test.columns:
    df_test['tx_per_month'] = df_test['num_transactions_30d'] / (df_test['tenure_months'] + 1)
    features_created += 1

if 'avg_amount_30d_eur' in df_test.columns and 'annual_income_eur' in df_test.columns:
    df_test['tx_to_income_ratio'] = df_test['avg_amount_30d_eur'] / (df_test['annual_income_eur'] / 12 + 1)
    features_created += 1

if 'chargebacks_12m' in df_test.columns and 'num_transactions_30d' in df_test.columns:
    df_test['chargeback_rate'] = df_test['chargebacks_12m'] / (df_test['num_transactions_30d'] * 4 + 1)
    features_created += 1

if 'failed_payments_6m' in df_test.columns and 'num_transactions_30d' in df_test.columns:
    df_test['failed_payment_rate'] = df_test['failed_payments_6m'] / (df_test['num_transactions_30d'] * 2 + 1)
    features_created += 1

risk_cols = [c for c in ['chargebacks_12m', 'failed_payments_6m', 'support_tickets_90d',
             'is_vpn', 'is_new_device'] if c in df_test.columns]
if len(risk_cols) > 0:
    risk_df_test = df_test[risk_cols].copy()
    for col in risk_cols:
        if df_test[col].std() > 0:
            risk_df_test[col] = (df_test[col] - df_test[col].mean()) / df_test[col].std()
    df_test['composite_risk_score'] = risk_df_test.sum(axis=1)
    features_created += 1

print(f"    ✓ {features_created} nouvelles features créées")


4️⃣  Feature Engineering...
    ✓ 5 nouvelles features créées


## 5. Encodage catégorielles

In [66]:
print("\n5️⃣  Encodage des variables catégorielles...")
label_encoders = {}
for col in categorical_cols:
    if col in df_test.columns:
        if df_test[col].nunique() > 50:
            top_values = df_test[col].value_counts().head(50).index
            df_test[col] = df_test[col].apply(lambda x: x if x in top_values else 'Other')
        
        le = LabelEncoder()
        df_test[col] = le.fit_transform(df_test[col].astype(str))
        label_encoders[col] = le

print(f"    ✓ {len(categorical_cols)} colonnes encodées")



5️⃣  Encodage des variables catégorielles...
    ✓ 2 colonnes encodées


## 6. Traitement des outliers

In [67]:
print("\n6️⃣  Traitement des outliers...")
numeric_cols = df_test.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in exclude_cols and c != target ]

for col in numeric_cols:
    if col in df_test.columns:
        q1 = df_test[col].quantile(0.01)
        q99 = df_test[col].quantile(0.99)
        df_test[col] = df_test[col].clip(lower=q1, upper=q99)
print(f"    ✓ Outliers traités")


6️⃣  Traitement des outliers...
    ✓ Outliers traités


## 7. Décision finale

In [68]:
features = [c for c in df_test.columns if c not in exclude_cols ]  
df_features = df_test[features]

df_processed = df_features.copy()
if 'customer_id' in df_test.columns:
    df_processed['customer_id'] = df_test['customer_id']

## 8. Export

In [69]:
print("\n💾 Export du dataset preprocessé...")
df_processed.to_csv('../data/data_preprocess_rf_test_v3.csv', index=False)
print("✅ data_preprocess_rf_test.csv créé!")


💾 Export du dataset preprocessé...
✅ data_preprocess_rf_test.csv créé!


In [70]:
df_processed

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,signup_year,signup_month,signup_day_of_week,days_since_signup,tx_per_month,tx_to_income_ratio,chargeback_rate,failed_payment_rate,composite_risk_score,customer_id
0,43,53,25587.55,639,24,106.61,781.55,7.3,0,0,...,2024,9,1,525,0.444444,0.049974,0.0,0.000000,0.136592,CUST_E5RX1BC9II
1,22,5,45378.40,699,21,54.17,252.95,4.2,1,0,...,2025,6,6,254,3.500000,0.014321,0.0,0.000000,-1.356902,CUST_BHWIUKERGN
2,42,2,36643.70,765,17,44.40,238.79,7.0,1,0,...,2024,8,6,562,5.666667,0.014535,0.0,0.000000,-1.356902,CUST_EXT9NA4CHU
3,39,20,30283.30,573,29,38.30,321.44,28.3,0,0,...,2025,5,3,299,1.380952,0.015171,0.0,0.016949,-0.794326,CUST_9FSJE5R1NY
4,54,10,35294.22,624,21,70.93,540.48,21.7,0,0,...,2024,3,0,715,1.909091,0.024108,0.0,0.000000,-2.466131,CUST_GDQXMODBED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,44,3,41531.58,622,24,42.99,214.60,7.3,0,0,...,2025,2,6,373,6.000000,0.012418,0.0,0.020408,-0.794326,CUST_1OM9UCID91
39996,48,12,36783.80,697,25,21.17,69.19,3.8,1,0,...,2024,7,3,607,1.923077,0.006904,0.0,0.000000,-1.356902,CUST_VDEY72BIZP
39997,49,28,55963.60,658,18,180.58,410.17,3.6,2,0,...,2025,12,3,82,0.620690,0.038713,0.0,0.000000,-0.247672,CUST_UQEZ9KKIFG
39998,31,7,9638.61,683,19,129.93,643.13,13.9,0,0,...,2025,5,4,291,2.375000,0.131449,0.0,0.000000,-2.466131,CUST_IXX0BE9VQD
